In [ ]:
%matplotlib inline

In [ ]:
# Imports
import os
import pathlib
import urllib.request

# Site name for Salvus Flow. Uses env var if available, otherwise falls back to local site name.
SALVUS_FLOW_SITE_NAME = os.environ.get("SALVUS_FLOW_SITE_NAME", "salome_remote_2")
PROJECT_DIR = "simulation_fault_zone" 

# Conservative default to reduce license-seat demand during unstable license-server periods.
RANKS_PER_JOB = 4


def check_license_server_reachable(url="https://l.mondaic.com/licensing_server", timeout=10):
    """Fail fast if licensing endpoint is unreachable to avoid long hanging jobs."""
    try:
        with urllib.request.urlopen(url, timeout=timeout):
            return True
    except Exception as exc:
        raise RuntimeError(
            f"Licensing server not reachable ({url}). Retry later or check network/VPN. Original error: {exc}"
        ) from exc


# Add code to keep .gitignore updated to ignore salvus files
gitignore_path = pathlib.Path("..") / ".gitignore"
with open(gitignore_path, "r+") as f:
    contents = f.read()
    if PROJECT_DIR not in contents:
        f.write(f"\n{PROJECT_DIR}/\n")


import numpy as np
import salvus.namespace as sn
import xarray as xr
import salvus.namespace as sn
from salvus.project.tools.processing import block_processing
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import matplotlib.pyplot as plt
from matplotlib import animation
import obspy

# For wavefield output code
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers import wavefield_output

#for plotting of wiggles, traces 
from scipy import signal

# For animation plot
from IPython.display import HTML
from matplotlib.collections import PolyCollection
import threading
import matplotlib
matplotlib.use("Agg")
from scipy.interpolate import griddata

In [ ]:
# Setup of the model domain as a box (same as research module)
domain_2d = sn.domain.dim2.BoxDomain(x0=0, x1=300, y0=0, y1=3)
p = sn.Project.from_domain(path=PROJECT_DIR, domain=domain_2d, load_if_exists=True)

In [ ]:
# Layered model setup: three layers ordered as snow, weak layer (wl), air.

x_min, x_max = 0.0, 300.0

# Geometry (high y = top of domain ):
slab_bottom = 0.0
slab_thickness = 1.0
wl_thickness = 0.05
domain_top = 3.0

slab_top = slab_bottom + slab_thickness # snow-wl interface
wl_bottom = slab_top
wl_top = wl_bottom + wl_thickness # wl-air interface
air_top = domain_top

# Boundaries from top to bottom -> 3 layers.
layers_x = [
    np.array([x_min, x_max]),  # top boundary
    np.array([x_min, x_max]),  # air-wl interface
    np.array([x_min, x_max]),  # wl-snow interface
    np.array([x_min, x_max]),  # bottom boundary
]
layers_y = [
    np.array([air_top, air_top]),
    np.array([wl_top, wl_top]),
    np.array([wl_bottom, wl_bottom]),
    np.array([slab_bottom, slab_bottom]),
]

# Material parameters by region index [snow, weak layer, air].
vp = np.array([300.0, 120.0, 332.0])
vs = np.array([150.0, 60.0, 0.0])
rho = np.array([180.0, 150.0, 1.225])

interpolation_styles = ["linear"] * len(layers_x)
splines = sn.toolbox.get_interpolating_splines(layers_x, layers_y, kind=interpolation_styles)

max_frequency = 50.0
# One per layer pair; last value keeps meshing stable above acoustic air.
slowest_velocities = np.array([150.0, 60.0, 60.0])

mesh, bnd = sn.toolbox.generate_mesh_from_splines_2d(
    x_min=x_min,
    x_max=x_max,
    splines=splines,
    elements_per_wavelength=2,
    maximum_frequency=max_frequency,
    use_refinements=True,
    slowest_velocities=slowest_velocities,
    absorbing_boundaries=(["x0", "x1", "y0"], 10.0),
)
mesh = np.sum(mesh)
mesh.attach_global_variable("max_dist_ABC", bnd)
mesh.attach_global_variable("ABC_side_sets", ", ".join(["x0", "x1", "y0"]))
mesh.attach_global_variable("ABC_vel", float(min(vs[vs > 0])))
mesh.attach_global_variable("ABC_freq", max_frequency / 2.0)
mesh.attach_global_variable("ABC_nwave", 5.0)

nodes = mesh.get_element_nodes()[:, :, 0]
vp_a, vs_a, ro_a = np.ones((3, *nodes.shape))
for _i, (vp_val, vs_val, ro_val) in enumerate(zip(vp, vs, rho)):
    idx = np.where(mesh.elemental_fields["region"] == _i)
    vp_a[idx] = vp_val
    vs_a[idx] = vs_val
    ro_a[idx] = ro_val

for k, v in zip(["VP", "VS", "RHO"], [vp_a, vs_a, ro_a]):
    mesh.attach_field(k, v)

mesh_3layer = sn.toolbox.detect_fluid(mesh)
print("Three-layer mesh built.")
print(f"  Snow layer: y = [{slab_bottom:.2f}, {slab_top:.2f}] m, vs=150 m/s")
print(f"  Weak layer: y = [{wl_bottom:.2f}, {wl_top:.2f}] m, vs=60 m/s")
print(f"  Air layer:  y = [{wl_top:.2f}, {air_top:.2f}] m, vs=0 m/s")

In [ ]:
# Writing standard rupture format file: based on https://www.mondaic.com/case-studies/accurate-fault-planes-in-earthquake-ground-motion-modeling

def write_snow_crack_srf(
    filepath,
    crack_x_start=30.0,
    crack_x_end=270.0,
    crack_y=wl_bottom + wl_thickness / 2.0, # crack is in the mdidle of weak layer
    n_subfaults=120,
    rupture_speed=125.0,  # m/s - sub-Rayleigh (vs_weak=60, vs_slab=150)
    rise_time=0.02, # second s duration of slip at each subfault
    total_slip=0.01, # m : max slip (10 mm), realistic for PST
    mode_mix=0.7,# fraction of shear (Mode II) vs opening (Mode I): if this is 1.0, it's pure shear; if 0.0, it's pure openingh 
    f0=10.0, # Ricker center frequency
):
    """
    Write an SRF-like ASCII file for a 2D horizontal snow crack.
    
    In 2D Cartesian Salvus, we approximate the SRF concept by generating
    the subfault parameters analytically and writing them in a format that
    maps to MomentTensorPoint2D sources with per-subfault STFs.
    
    The function returns the list of (x, y, delay, mxx, myy, mxy, rise_time)
    tuples so they can be passed directly to Salvus.
    """
    crack_length = crack_x_end - crack_x_start
    dx = crack_length / (n_subfaults - 1)
    x_positions = np.linspace(crack_x_start, crack_x_end, n_subfaults)

    # Onset time: crack initiates at x_start, propagates at rupture_speed
    onset_times = (x_positions - crack_x_start) / rupture_speed

    # Slip distribution: elliptical (Kostrov- like), tapered at tips
    xi = (x_positions - crack_x_start) / crack_length  # 0 to 1
    slip_distribution = total_slip * np.sqrt(np.maximum(0, xi * (1 - xi)) * 4)

    # Moment tensor components (2D plane strain):
    # Mode II (shear): mxy - slip along x, interface normal is y
    # Mode I (opening): myy - normal opening
    # Scale by shear modulus of weak layer: mu = rho * vs^2
    mu_weak = 150.0 * 60.0**2    # ~540 kPa
    lam_weak = rho[1] * (vp[1]**2 - 2 * vs[1]**2)  # for Mode I

    subfaults = []
    for i, (x_src, t_onset, slip) in enumerate(zip(x_positions, onset_times, slip_distribution)):
        # Seismic moment per unit length (2D) = mu * slip * patch_width
        M0 = mu_weak * slip * dx

        # Mixed-mode moment tensor in 2D:
        # Mode II shear: Mxy component
        # Mode I opening: Myy component (tensile)
        mxy = mode_mix * M0
        myy = (1.0 - mode_mix) * M0
        mxx = 0.0  # no out-of-plane contribution in 2D

        subfaults.append({
            "x": float(x_src),
            "y": float(crack_y),
            "onset_time": float(t_onset),
            "rise_time": rise_time,
            "mxx": mxx,
            "myy": myy,
            "mxy": mxy,
            "slip": slip,
        })

    # Save to file for documentation
    with open(filepath, "w") as f:
        f.write("# Snow crack SRF-equivalent descriptor\n")
        f.write(f"# crack_y={crack_y:.4f} m, rupture_speed={rupture_speed} m/s\n")
        f.write(f"# n_subfaults={n_subfaults}, rise_time={rise_time} s\n")
        f.write("# x_m  y_m  onset_s  rise_s  mxx  myy  mxy  slip_m\n")
        for sf in subfaults:
            f.write(
                f"{sf['x']:.4f}  {sf['y']:.4f}  {sf['onset_time']:.6f}  "
                f"{sf['rise_time']:.4f}  {sf['mxx']:.4f}  {sf['myy']:.4f}  "
                f"{sf['mxy']:.4f}  {sf['slip']:.6f}\n"
            )

    print(f"Wrote {len(subfaults)} subfaults to {filepath}")
    print(f"  Crack length: {crack_length:.1f} m")
    print(f"  Rupture duration: {onset_times[-1]:.3f} s")
    print(f"  Peak slip: {slip_distribution.max():.4f} m")
    print(f"  Max M0 (per subfault): {max(abs(sf['mxy'])+abs(sf['myy']) for sf in subfaults):.2e} N")
    return subfaults


srf_path = pathlib.Path(PROJECT_DIR) / "snow_crack.srf_equiv"
srf_path.parent.mkdir(exist_ok=True)

subfaults = write_snow_crack_srf(
    srf_path,
    crack_x_start=30.0,
    crack_x_end=270.0,
    crack_y=wl_bottom + wl_thickness / 2.0,
    n_subfaults=120,
    rupture_speed=125.0, # sub-Rayleigh, change for supershera
    rise_time=0.02,
    total_slip=0.01,
    mode_mix=0.7,
    f0=10.0,
)

In [ ]:
# Using srf to build source array 

def build_salvus_sources_from_srf(subfaults, f0=10.0, pre_delay=0.3):
    """
    Convert the SRF subfault list into Salvus MomentTensorPoint2D sources,
    each with a Ricker STF delayed to match the subfault onset time.
    The rise_time modulates the effective STF width.
    """
    srcs = []
    for sf in subfaults:
        # Effective time shift: pre_delay + crack onset + half rise-time
        time_shift = pre_delay + sf["onset_time"] + sf["rise_time"] / 2.0
        
        src = sn.simple_config.source.cartesian.MomentTensorPoint2D(
            x=sf["x"],
            y=sf["y"],
            mxx=sf["mxx"],
            myy=sf["myy"],
            mxy=sf["mxy"],
            source_time_function=sn.simple_config.stf.Ricker(
                center_frequency=f0,
                time_shift_in_seconds=time_shift,
            ),
        )
        srcs.append(src)

    print(f"Built {len(srcs)} Salvus sources from SRF descriptor.")
    print(f"  STF time shifts span {pre_delay:.2f}s to {srcs[-1].source_time_function.time_shift_in_seconds:.3f}s")
    return srcs


crack_srcs = build_salvus_sources_from_srf(subfaults, f0=10.0, pre_delay=0.3)

crack_event_name = "event_snow_crack_srf"
if crack_event_name in p.events.list():
    p.events.delete(event_name=crack_event_name)

p.add_to_project(sn.Event(event_name=crack_event_name, sources=crack_srcs))
print(f"Added event '{crack_event_name}' with {len(crack_srcs)} sources.")

In [ ]:

crack_sim_name = "sim_snow_crack_3layer"

p.add_to_project(
    sn.UnstructuredMeshSimulationConfiguration(
        name=crack_sim_name,
        unstructured_mesh=mesh_3layer,
        event_configuration=sn.EventConfiguration(
            wavelet=sn.simple_config.stf.Delta(),
            waveform_simulation_configuration=sn.WaveformSimulationConfiguration(
                start_time_in_seconds=-0.3,
                end_time_in_seconds=3.0, # longer becasue crack takes abt 1.9s to traverse 240m at 125 m/s
            ),
        ),
    ),
    overwrite=True,
)

input_file = p.simulations.get_simulation_template(crack_sim_name, crack_event_name)
input_file.add_sources(crack_srcs)

# Validate source count
assert len(input_file.physics.wave_equation.point_source) == len(crack_srcs), \
    f"Source count mismatch: {len(input_file.physics.wave_equation.point_source)} vs {len(crack_srcs)}"

# Request volume and  surface output
del input_file.output.point_data

input_file.output.volume_data = {
    "filename": "volume_data_crack.h5",
    "format": "hdf5",
    "fields": ["velocity", "displacement", "strain"],
    "sampling_interval_in_time_steps": 50,
}

input_file.validate()

# Launch simulations 
crack_output_folder = str(pathlib.Path(PROJECT_DIR) / "job_snow_crack_3layer")
crack_job = sn.api.run(
    input_file=input_file,
    site_name=SALVUS_FLOW_SITE_NAME,
    ranks=RANKS_PER_JOB,
    output_folder=crack_output_folder,
    get_all=True,
    overwrite=True,
)
print(f"Run finished. Output: {crack_output_folder}")